In [1]:
# Import Libraries
import pandas as pd
import numpy as np

In [2]:
src = 'clean.csv'
df = pd.read_csv(src, sep=',', encoding='utf-8')

# Initial Data Loading

In [3]:
df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 109 columns):
 #    Column                                          Non-Null Count  Dtype  
---   ------                                          --------------  -----  
 0    Health Center                                   300 non-null    str    
 1    Gender                                          298 non-null    str    
 2    Age                                             300 non-null    str    
 3    Weight                                          196 non-null    float64
 4    Mua circumference                               25 non-null     str    
 5    Temperature                                     299 non-null    str    
 6    Fever 48 hrs                                    297 non-null    str    
 7    Fever in the last 7 days                        296 non-null    str    
 8    Type of fever                                   294 non-null    str    
 9    Loss of Weight                           

In [4]:
df.head()

,Health Center,Gender,Age,Weight,Mua circumference,Temperature,Fever 48 hrs,Fever in the last 7 days,Type of fever,Loss of Weight,...,Malaria,Diagnosed Dengue,Diagnosed Chikungunya,Yellow fever,Typhoid fever,Diagnosed Zika,Other diagnosed diseases,Diagnosed Option 8,Other diseases presented by patient,UUID
0,CMA DO,Female,18,55.0,NaN,YES,YES,YES,Recurrent,NO,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,df22d899-23a3-4c49-8059-6208c130f57d
1,CMA DO,Male,26,120.0,NaN,YES,YES,YES,Recurrent,NO,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,4a7ef2f3-4ff0-4215-8cb5-20ba8c8d4355
2,CMA DO,Male,25,63.0,NaN,YES,YES,YES,Recurrent,NO,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,NaN,104f6bc8-f39a-482b-b680-7ac1f8cccc86
3,CMA DO,Male,3,12.0,"13,5",YES,YES,NO,Recurrent,NO,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,26d7d04a-1a1c-4d4b-b64a-606f968e6d9c
4,CMA DO,Male,8,24.0,NaN,YES,YES,NO,Recurrent,NO,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,eb694f52-51ff-488d-9beb-2923a572a4cf


In [5]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": df.isnull().sum() / len(df)
})


In [6]:
print(missing[missing["Missing Values"] > 0].to_string())

                                                Missing Values  Percentage
Gender                                                       2    0.006667
Weight                                                     104    0.346667
Mua circumference                                          275    0.916667
Temperature                                                  1    0.003333
Fever 48 hrs                                                 3    0.010000
Fever in the last 7 days                                     4    0.013333
Type of fever                                                6    0.020000
Loss of Weight                                               2    0.006667
Headache                                                     2    0.006667
Nausea                                                       3    0.010000
Vomiting                                                     1    0.003333
Muscle pain                                                  1    0.003333
Joint Swelling           

# Data Cleansing

In [7]:
df_temp = df.copy()

In [8]:
target_cols = ['Malaria', 'Diagnosed Dengue', 'Yellow fever', 
               'Typhoid fever', 'Other diagnosed diseases']

num_issues = ['Hematocrit', 'White blood cell count', 'Platelet count', 'Neutrophils', 'Age']

redudant_cols = [
    'UUID', 'Diagnosed diseases', 'Other diseases presented by patient',
    'Diagnosed Chikungunya', 'Diagnosed Zika', 'Diagnosed Option 8',
    'Mua circumference', 'Capillary refill time', 'Arterial blood pressure',
    'Lymphocytes', 'Health Center', 'Conjunctivitis', 'Facial flushing', 'Profuse sweating',
    'Rheumatic disease', 'Autoimmune disease', 'Allergies', 'Cancer', 'Leucopenia', 
]


disease_to_target = {
    'Malaria':  'Malaria',
    'Dengue':   'Diagnosed Dengue',
    'Yellow':   'Yellow fever',
    'Typhoid':  'Typhoid fever',
}



In [9]:
# 1. Fixing Formatting Issues
for col in num_issues:
    df_temp[col] = df_temp[col].astype(str).str.replace(',', '.', regex=False)
    df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce')
    
def clean_pulse(val):
    if pd.isna(val): return np.nan
    if '/' in str(val) or '|' in str(val): return np.nan
    try: return float(val)
    except: return np.nan

df_temp['Pulse rate'] = df_temp['Pulse rate'].apply(clean_pulse)

In [10]:
# 2. Parsing Diagnosed Diseasies to backfill targets
def parse_diagnosed_diseases(text):
    if pd.isna(text) or str(text).strip() == '':
        return {col: np.nan for col in target_cols}
    return {
        'Malaria':                  1 if 'Paludisme' in text or 'Malaria' in text else 0,
        'Diagnosed Dengue':         1 if 'Dengue' in text else 0,
        'Yellow fever':             1 if 'jaune' in text or 'yellow' in text.lower() else 0,
        'Typhoid fever':            1 if 'Typho' in text else 0,
        'Other diagnosed diseases': 1 if 'Autres' in text or 'Others' in text else 0
    }

parsed = pd.DataFrame(
    df_temp['Diagnosed diseases'].apply(parse_diagnosed_diseases).to_list(),
    index=df_temp.index
)

for col in target_cols:
    mask = df_temp[col].isna()
    df_temp.loc[mask, col] = parsed.loc[mask, col]


In [11]:
# 3. Drop Blank-Diagnosis Rows
blank_mask = (
    df_temp['Diagnosed diseases'].isna() | 
    (df_temp['Diagnosed diseases'].str.strip() == '')
)
print(f"Dropping {blank_mask.sum()} rows with no diagnosis")
df_temp = df_temp[~blank_mask].reset_index(drop=True)

# Any targets still NaN after backfill → assume 0
for col in target_cols:
    df_temp[col] = df_temp[col].fillna(0)

Dropping 1 rows with no diagnosis


In [12]:
# 4. Droping Redundant Columns
df_temp.drop(columns=redudant_cols, inplace=True)

In [13]:
# 5. Categorical Encoding
binary_cols = [col for col in df_temp.columns if df_temp[col].dropna().isin(['YES', 'NO']).all()]

for col in binary_cols:
    df_temp[col] = df_temp[col].map({'YES': 1, 'NO':0})
    
df_temp['Gender']           = df_temp['Gender'].map({'Male': 1, 'Female': 0})
df_temp['Type of fever']    = df_temp['Type of fever'].map({'Recurrent': 1, 'Intermittent': 0})
df_temp['RDT Test']         = df_temp['RDT Test'].map({'Positive': 1, 'Negative': 0})
df_temp['Thick blood smear']= df_temp['Thick blood smear'].map({'Positive': 1, 'Negative': 0})


In [14]:
def clean_temperature(val):
    if pd.isna(val): return np.nan
    val = str(val).strip().replace('T', '').replace(',', '.')
    try:
        temp = float(val)
        # Fix missing decimal point — all valid temps are 35-42
        # so any 4-digit integer like 3809 → 38.09, 3501 → 35.01
        if temp >= 3500:
            temp = temp / 100
        # Fix 3-digit integers like 390 → 39.0, 367 → 36.7
        elif temp >= 350:
            temp = temp / 10
        # Valid range check
        if 35.0 <= temp <= 42.5:
            return temp
        return np.nan
    except:
        return np.nan

df_temp['Axillary temperature'] = df_temp['Axillary temperature'].apply(clean_temperature)

# Verify
print(df_temp['Axillary temperature'].describe())
print(f"NaN count: {df['Axillary temperature'].isna().sum()}")

count    259.000000
mean      38.038764
std        1.220860
min       35.010000
25%       37.000000
50%       38.000000
75%       39.000000
max       40.900000
Name: Axillary temperature, dtype: float64
NaN count: 37


In [15]:
# Save Checkpoint
df_clean = df_temp.copy()
df_clean.to_csv('clean_base.csv', index=False)
print(f"Base cleaning done. Shape: {df_clean.shape}")

Base cleaning done. Shape: (299, 90)


# Data Prepping

In [16]:
def make_label(row):
    labels = []
    if row['Malaria'] == 1:                  labels.append('Malaria')
    if row['Diagnosed Dengue'] == 1:         labels.append('Dengue')
    if row['Yellow fever'] == 1:             labels.append('Yellow Fever')
    if row['Typhoid fever'] == 1:            labels.append('Typhoid')
    if row['Other diagnosed diseases'] == 1: labels.append('Other')
    return ' + '.join(labels) if labels else 'Unknown'

In [17]:
symptom_groups = {
    'hemorrhagic_score': ['Bleeding', 'Epistaxis (Bleeding nose)',
                          'Positive tourniquet test', 'Hemoglobinuria'],
    'neuro_score':       ['Consciousness trouble', 'Delirium', 'Irrational talking',
                          'Generalised or focal convulsion', 'Multiple convulsions',
                          'Impaired level of consciousness', 'Drowsiness or lethargy'],
    'gi_score':          ['Nausea', 'Vomiting', 'Diarrhea', 'Abdominal pain',
                          'Abdominal distension', 'Loss of appetite or Anorexia'],
    'musculo_score':     ['Muscle pain', 'Joint pain', 'Joint Swelling',
                          'Rachiodynia', 'Stiffness'],
    'systemic_score':    ['Fever 48 hrs', 'Fever in the last 7 days',
                          'Shiver or cold sensation', 'Prostration'],
    'respiratory_score': ['Cough', 'Respiratory distress', 'Rhinorrhea',
                          'Chest pain', 'Throat pain'],
    'severity_score':    ['Shock', 'Myocarditis', 'Septicemia',
                          'Accumulation of fluid and respiratory distress',
                          'Hepatosplenomegaly'],
}


In [18]:
def impute_by_group(df, group_col, group_medians, global_medians, cols):
    df = df.copy()
    for group, group_df in df.groupby(group_col):
        for col in cols:
            if df.loc[group_df.index, col].isna().any():
                # Use group median if available, else global
                fill_val = (
                    group_medians.loc[group, col]
                    if group in group_medians.index
                    else global_medians[col]
                )
                df.loc[group_df.index, col] = (
                    df.loc[group_df.index, col].fillna(fill_val)
                )
    return df

## Analytics

In [19]:
df_viz = df_clean.copy()

## Modelling

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

In [21]:
df_prep = df_clean.copy()

In [22]:
# 1. Seperating X & Y
y = df_prep[target_cols].copy()
X = df_prep.drop(columns=target_cols)

In [23]:
# 2. Multi-label stratified splitting
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
for train_idx, test_idx in msss.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

msss_val = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
for train_idx, val_idx in msss_val.split(X_train, y_train):
    X_val   = X_train.iloc[val_idx]
    y_val   = y_train.iloc[val_idx]
    X_train = X_train.iloc[train_idx]
    y_train = y_train.iloc[train_idx]

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (214, 85), Val: (37, 85), Test: (48, 85)


In [24]:
# 3. Diseased-informed binary imputation
train_combined = X_train.copy()
train_combined['_disease_group'] = (
    y_train[target_cols].apply(make_label, axis=1)
)

group_medians = (
    train_combined.groupby('_disease_group')[binary_cols].median()
)
global_medians = X_train[binary_cols].median()

X_train = impute_by_group(
    train_combined, '_disease_group',
    group_medians, global_medians, binary_cols
)
X_train.drop(columns=['_disease_group'], inplace=True)

for split in [X_val, X_test]:
    for col in binary_cols:
        split[col] = split[col].fillna(global_medians[col])



In [25]:
# 4. Imputing Numerical Columns
num_cols = ['Weight', 'Respiratory rate', 'Pulse rate', 'Elevated Creatinine',
            'Hematocrit', 'White blood cell count', 'Platelet count', 'Neutrophils',
            'Age', 'Axillary temperature']

binary_cols = [
    col for col in X_train.columns
    if col not in num_cols
    and X_train[col].dropna().isin([0, 1, 0.0, 1.0]).all()
]

num_imputer = SimpleImputer(strategy='median')
bin_imputer = SimpleImputer(strategy='most_frequent')

num_imputer.fit(X_train[num_cols])
bin_imputer.fit(X_train[binary_cols])

for split in [X_train, X_val, X_test]:
    split[num_cols] = num_imputer.transform(split[num_cols])
    split[binary_cols] = bin_imputer.transform(split[binary_cols])


In [26]:
# 5. Winsorizing + Log Transform
clip_caps = {}
log_cols = ['Elevated Creatinine', 'Respiratory rate',
            'White blood cell count', 'Platelet count']

for col in num_cols:
    cap = X_train[col].quantile(0.99)
    clip_caps[col] = cap
    for split in [X_train, X_val, X_test]:
        split[col] = split[col].clip(upper=cap)

for col in log_cols:
    for split in [X_train, X_val, X_test]:
        split[col] = np.log1p(split[col])


In [27]:
# 6. Scale (fit only done on train)
scaler = RobustScaler()
scaler.fit(X_train[num_cols])

for split in [X_train, X_val, X_test]:
    split[num_cols] = scaler.transform(split[num_cols])

In [28]:
# 7. Feature Engineering on all splits
def add_scores(df):
    for score_name, cols in symptom_groups.items():
        df[score_name] = df[cols].sum(axis=1)
    df['total_symptom_count'] = sum(df[s] for s in symptom_groups)

    comorbidity_cols = [ 'Osteoarthritis', 'Asthma', 'Pneumonia', 'Hypertension']
    df['comorbidity_count'] = df[comorbidity_cols].sum(axis=1)

    all_grouped = [c for cols in symptom_groups.values() for c in cols] + comorbidity_cols
    df.drop(columns=all_grouped, inplace=True)
    return df

X_train = add_scores(X_train)
X_val   = add_scores(X_val)
X_test  = add_scores(X_test)

In [29]:
# 8. Feature Selection (fitting on train)
selected_features = set()
for col in target_cols:
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train[col])
    importance = pd.Series(rf.feature_importances_, index=X_train.columns)
    top = importance.nlargest(20).index.tolist()
    selected_features.update(top)

selected_features = list(selected_features)
X_train = X_train[selected_features]
X_val   = X_val[selected_features]
X_test  = X_test[selected_features]

In [30]:
# 9. SMOTE on train only
smote = SMOTE(random_state=42)
X_train_res, y_train_malaria = smote.fit_resample(X_train, y_train['Malaria'])